In [5]:
import pandas as pd

In [7]:
# Load the survey and user info datasets
survey = pd.read_csv("../data/raw/DANTE_Pilot_October 13, 2025_21.48.csv")
user_info = pd.read_csv("../data/raw/User_Info.csv")

# Drop the first two rows which contain metadata
survey = survey.drop(index=[0,1])

# Rename ParticipantId to participantId for consistency across datasets
user_info.rename(columns={'ParticipantId': 'participantId'}, inplace=True)

# Drop the 'Status' column as it exists in the survey dataframe
user_info = user_info.drop(columns=['Status'])

# Merge the survey and user_info dataframes on participantId
survey = survey.merge(user_info, on="participantId", how="left")

# Rename 'initial_opinion' to 'user_response_1' and drop the original column
survey['user_response_1'] = survey['initial_opinion']
survey.drop(columns=['initial_opinion'], inplace=True)

In [8]:
# Drop if participant ID is non-existent - internal test cases
survey = survey[(~survey['participantId'].isna()) & (survey['participantId']!='')]

# Remove partial completes
survey = survey[(survey['CompletionCode'] != "12FB757AF3") & (~survey['CompletionCode'].isna())]

# Convert 'Finished' column to boolean
survey['Finished'] = survey['Finished'].apply(lambda x: eval(x))

# Keep only finished surveys
survey = survey[survey['Finished'] == True]

# Remove participants we couldn't generate summaries for
survey = survey[(~survey['summary'].isna()) & (survey['summary']!='')]

# Remove treatment block errors where LLM response is missing
survey = survey[(~survey['llm_response_1'].isna()) & (survey['llm_response_1']!='')]

In [9]:
def remove_incomplete_llm_interactions(df):
    """
    Remove participants who sent a user response but didn't receive a corresponding LLM response.
    
    Checks pairs in order: user_response_1 -> llm_response_1, user_response_2 -> llm_response_2, etc.
    
    Args:
        df: DataFrame containing user_response and llm_response columns
        
    Returns:
        DataFrame with incomplete interactions removed
    """
    mask = pd.Series([True] * len(df), index=df.index)
    
    for i in range(1, 6):  # Check responses 1 through 5
        user_col = f'user_response_{i}'
        llm_col = f'llm_response_{i}'
        
        # If user sent a response but didn't get LLM response back, mark for removal
        has_user_response = (~df[user_col].isna()) & (df[user_col] != '')
        missing_llm_response = (df[llm_col].isna()) | (df[llm_col] == '')
        
        incomplete = has_user_response & missing_llm_response
        mask = mask & (~incomplete)
    
    removed_count = (~mask).sum()
    print(f"Found and removed {removed_count} participants with incomplete LLM interactions")
    
    return df[mask]

# Apply the function
survey = remove_incomplete_llm_interactions(survey)

survey = survey.drop_duplicates(subset=['IPAddress'],keep=False)

Found and removed 24 participants with incomplete LLM interactions


In [10]:
survey

,StartDate,EndDate,Status,IPAddress,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,RecipientLastName,...,Sex,Occupation Field,Relationship/Marital Status,Political Party,Ethnicity,Gender,Country Of Residence,Household Income,Race,Employment Status
0,2025-09-17 10:07:34,2025-09-17 10:12:03,IP Address,98.122.207.40,100,269,True,2025-09-17 10:12:03,R_7oCsprcktCBtsVG,NaN,...,Female,I’d Rather Not Say,Married,Democrat,"No, not of Hispanic, Latino, or Spanish origin",Woman,United States,"$20,000-$29,999",White,Unemployed
1,2025-09-17 10:08:34,2025-09-17 10:12:53,IP Address,68.81.204.147,100,258,True,2025-09-17 10:12:53,R_5y2Qdrp064Ag88S,NaN,...,Female,"Science, Technology, Engineering & Mathematics",In a relationship,Democrat,"No, not of Hispanic, Latino, or Spanish origin",Woman,United States,"$100,000-$124,999",Other,Full-time
2,2025-09-17 10:08:02,2025-09-17 10:14:27,IP Address,64.226.154.114,100,384,True,2025-09-17 10:14:27,R_5czsPmFlqnSZ2iP,NaN,...,Female,Education & Training,Married,Democrat,"No, not of Hispanic, Latino, or Spanish origin",Woman,United States,"$90,000-$99,999",White,Full-time
3,2025-09-17 10:07:50,2025-09-17 10:15:35,IP Address,24.254.250.19,100,464,True,2025-09-17 10:15:35,R_7H8KDxgopCfxgsx,NaN,...,Female,Finance,In a relationship,Republican,"No, not of Hispanic, Latino, or Spanish origin",Woman,United States,"$80,000-$89,999",White,Student
4,2025-09-17 10:08:07,2025-09-17 10:15:45,IP Address,166.198.28.65,100,458,True,2025-09-17 10:15:45,R_5R7x26yCIF45oYu,NaN,...,Male,Government & Public Administration,Married,Democrat,"No, not of Hispanic, Latino, or Spanish origin",Man,United States,"$70,000-$79,999",White,Full-time
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2075,2025-10-11 07:26:46,2025-10-11 07:35:26,IP Address,173.67.31.173,100,519,True,2025-10-11 07:35:27,R_360f2Z17lb4AC1X,NaN,...,Male,Information Technology,Married,Democrat,"No, not of Hispanic, Latino, or Spanish origin",Man,United States,"$225,000-$249,999",White,Full-time
2076,2025-10-11 07:16:30,2025-10-11 07:35:40,IP Address,172.59.112.38,100,1149,True,2025-10-11 07:35:40,R_1o5c6bdqEvNWC2D,NaN,...,Male,Military,Married,Democrat,"No, not of Hispanic, Latino, or Spanish origin",Man,United States,"$90,000-$99,999",Black or African American,Full-time
2077,2025-10-11 07:10:24,2025-10-11 07:37:16,IP Address,68.235.139.47,100,1612,True,2025-10-11 07:37:17,R_79cvcoCuFggB2zO,NaN,...,Male,"Science, Technology, Engineering & Mathematics",Married,Democrat,"Yes, another Hispanic, Latino, or Spanish orig...",Man,United States,"$90,000-$99,999",White,Full-time
2078,2025-10-11 07:28:47,2025-10-11 07:45:17,IP Address,24.188.27.245,100,989,True,2025-10-11 07:45:18,R_7VIIRy9o89QKiUH,NaN,...,Male,Other,Single,Democrat,"No, not of Hispanic, Latino, or Spanish origin",Man,United States,"$40,000-$49,999",White,Full-time
